# ДЗ 2. Функции и словари на настоящих данных
Срок: **до понедельника 19:00**.

**Сдача через GitHub** (инструкция «Git для домашних заданий»): сохраните ноутбук как `hw02.ipynb` в папку `hw02` репозитория `dsa-homework`, затем `git add hw02` → `git commit -m "hw02: ..."` → `git push`. Из Colab: Файл → Скачать → Скачать .ipynb. Файл `orders.csv` в репозиторий класть не нужно.

Файл `orders.csv` — заказы «Дала Маркет» за сентябрь и октябрь 2026. Работаем **без Pandas и NumPy**: только `csv`, функции, словари, множества.

- **[База]** обязательно для всех.
- **[Стандарт]** решайте, сколько получится.
- **[Челлендж]** для тех, кому легко.

**Правило AI:** можно спросить, *почему* не работает код, но нельзя просить решение.

## [Для всех] Чтение кода — без запуска
Для каждого фрагмента: **сначала** напишите в Markdown-ячейке ответ и одно предложение «почему», **потом** запустите и сравните. Ошибка в предсказании не снижает оценку — снижает отсутствие объяснения.
```python
# 1
r = {}
for c in ["A", "B", "A"]:
    r[c] = r.get(c, 0) + 1
print(r)

# 2
def f(x):
    x.append(1)
a = [0]; f(a); f(a); print(a)

# 3
s = {c.lower() for c in ["Almaty", "ALMATY", "Astana"]}; print(len(s))

# 4
d = {k: len(k) for k in ["ab", "abc", "ab"]}; print(d)

# 5
def g(n):
    if n > 0:
        print(n)
x = g(5); print(x)
```
На следующем занятии несколько человек объяснят вслух одну строку своего ДЗ.

Мои предсказания и объяснения:

1. `{'A': 2, 'B': 1}` — словарь считает, сколько раз встретилась каждая буква.
2. `[0, 1, 1]` — функция дважды изменяет один и тот же список через `append(1)`.
3. `2` — после `lower()` значения `Almaty` и `ALMATY` становятся одинаковыми, а `Astana` остаётся вторым уникальным элементом.
4. `{'ab': 2, 'abc': 3}` — повторный ключ `ab` перезаписывает то же значение, поэтому ключ в словаре один.
5. Сначала напечатает `5`, затем `None` — функция ничего не возвращает явно, поэтому результат вызова равен `None`.


In [ ]:
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [ ]:
import csv
with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))
print(len(rows), "строк")

5463 строк


## [База]
1. Напишите функции `to_num(s)` и `clean_city(s)` (6 городов, латиница и разный регистр → кириллица с заглавной).
2. Сколько заказов в каждой категории (`category`)? Отсортируйте по убыванию.
3. Для каждой категории посчитайте выручку оплаченных заказов и средний чек. Выведите таблицу f-строкой.
4. Сколько заказов в каждом месяце? Месяц — это `order_date[:7]`.

In [ ]:

def to_num(s):
    return float(str(s).strip().replace(' ', '').replace(',', '.'))

def clean_city(s):
    city = str(s).strip().lower()
    cities_map = {
        'алматы': 'Алматы', 'almaty': 'Алматы',
        'астана': 'Астана', 'astana': 'Астана',
        'шымкент': 'Шымкент', 'shymkent': 'Шымкент',
        'караганда': 'Караганда', 'karaganda': 'Караганда',
        'актобе': 'Актобе', 'aktobe': 'Актобе',
        'атырау': 'Атырау', 'atyrau': 'Атырау'
    }
    return cities_map.get(city, city.title())

for row in rows:
    row['amount_num'] = to_num(row['amount'])
    row['city_clean'] = clean_city(row['city'])

category_count = {}
for row in rows:
    category = row['category']
    category_count[category] = category_count.get(category, 0) + 1

print('Заказы по категориям:')
for category, count in sorted(category_count.items(), key=lambda item: item[1], reverse=True):
    print(f'{category}: {count}')

category_revenue = {}
category_paid_count = {}
for row in rows:
    if row['status'] == 'оплачен':
        category = row['category']
        category_revenue[category] = category_revenue.get(category, 0) + row['amount_num']
        category_paid_count[category] = category_paid_count.get(category, 0) + 1

print('\nКатегория | Выручка оплаченных | Средний чек')
for category in sorted(category_revenue, key=category_revenue.get, reverse=True):
    average = category_revenue[category] / category_paid_count[category]
    print(f'{category} | {category_revenue[category]:,.0f} | {average:,.2f}')

month_count = {}
for row in rows:
    month = row['order_date'][:7]
    month_count[month] = month_count.get(month, 0) + 1

print('\nЗаказы по месяцам:')
for month in sorted(month_count):
    print(f'{month}: {month_count[month]}')


Заказы по категориям:
Одежда: 1529
Электроника: 1185
Дом: 1105
Красота: 991
Книги: 653

Категория | Выручка оплаченных | Средний чек
Электроника | 65,270,240 | 61,286.61
Одежда | 18,321,080 | 13,285.77
Дом | 10,516,320 | 10,654.83
Красота | 5,507,620 | 6,188.34
Книги | 2,294,100 | 3,934.99

Заказы по месяцам:
2026-09: 2782
2026-10: 2681


## [Стандарт]
5. Выручка оплаченных по месяцам и изменение октября к сентябрю в процентах.
6. Доля отменённых заказов по каждому (чистому) городу, в процентах с одним знаком.
7. Топ-5 клиентов (`customer_id`) по выручке оплаченных заказов. Кто на первом месте и что с ним не так? Ответьте в Markdown-ячейке.
8. Сколько **уникальных** клиентов в каждом городе? Подсказка: словарь множеств.

**Ответ к задаче 7:** На первом месте клиент `TEST` с выручкой 29 999 970 ₸. Это подозрительная тестовая запись: у него три заказа примерно по 9 999 990 ₸ каждый, поэтому он сильно искажает общую выручку.


In [ ]:
month_revenue = {}
for row in rows:
    if row['status'] == 'оплачен':
        month = row['order_date'][:7]
        month_revenue[month] = month_revenue.get(month, 0) + row['amount_num']

for month in sorted(month_revenue):
    print(f'{month}: {month_revenue[month]:,.0f} ₸')
change_percent = (month_revenue['2026-10'] - month_revenue['2026-09']) / month_revenue['2026-09'] * 100
print(f'Изменение октября к сентябрю: {change_percent:.2f}%')

city_total = {}
city_cancelled = {}
for row in rows:
    city = row['city_clean']
    city_total[city] = city_total.get(city, 0) + 1
    if row['status'] == 'отменён':
        city_cancelled[city] = city_cancelled.get(city, 0) + 1

print('\nГород | Отменено, %')
for city in sorted(city_total):
    percent = city_cancelled.get(city, 0) / city_total[city] * 100
    print(f'{city} | {percent:.1f}%')

customer_revenue = {}
for row in rows:
    if row['status'] == 'оплачен':
        customer = row['customer_id']
        customer_revenue[customer] = customer_revenue.get(customer, 0) + row['amount_num']

top_customers = sorted(customer_revenue.items(), key=lambda item: item[1], reverse=True)[:5]
print('\nТоп-5 клиентов:')
for customer, revenue in top_customers:
    print(f'{customer}: {revenue:,.0f} ₸')

city_customers = {}
for row in rows:
    city = row['city_clean']
    if city not in city_customers:
        city_customers[city] = set()
    city_customers[city].add(row['customer_id'])

print('\nУникальные клиенты по городам:')
for city in sorted(city_customers):
    print(f'{city}: {len(city_customers[city])}')


2026-09: 57,489,850 ₸
2026-10: 44,419,510 ₸
Изменение октября к сентябрю: -22.74%

Город | Отменено, %
Актобе | 10.3%
Алматы | 7.9%
Астана | 8.8%
Атырау | 7.5%
Караганда | 8.9%
Шымкент | 20.4%

Топ-5 клиентов:
TEST: 29,999,970 ₸
C01627: 257,180 ₸
C00485: 215,920 ₸
C01338: 192,080 ₸
C02122: 191,670 ₸

Уникальные клиенты по городам:
Актобе: 358
Алматы: 1351
Астана: 981
Атырау: 138
Караганда: 451
Шымкент: 655


## [Челлендж]
9. Найдите полные дубли строк: превратите каждую строку в `tuple(r.values())` и проверяйте через множество. Сколько их и в каком месяце?
10. Пересчитайте выручку по месяцам без дублей и без клиента из задачи 7. Как изменился процент из задачи 5?
11. У какого клиента больше всего заказов (после удаления дублей)? Сколько?
12. Напишите менеджеру 3–4 предложения: насколько на самом деле упала выручка и почему первый расчёт был неверным.

**Ответ к задаче 12:** В первом расчёте выручка октября оказалась ниже сентября на 22,7%. После удаления 180 полных дублей и исключения тестового клиента `TEST` снижение составило только 2,3%. Первый расчёт был искажён повторными строками и тремя аномально крупными тестовыми заказами клиента `TEST`. Поэтому для оценки реальной динамики нужно очищать данные от дублей и тестовых записей.


In [ ]:

seen_rows = set()
duplicate_rows = []
unique_rows = []
for row in rows:
    row_tuple = tuple(row.values())
    if row_tuple in seen_rows:
        duplicate_rows.append(row)
    else:
        seen_rows.add(row_tuple)
        unique_rows.append(row)

duplicate_months = {}
for row in duplicate_rows:
    month = row['order_date'][:7]
    duplicate_months[month] = duplicate_months.get(month, 0) + 1
print('Полных дублей:', len(duplicate_rows))
print('Дубли по месяцам:', duplicate_months)

clean_revenue = {}
for row in unique_rows:
    if row['status'] == 'оплачен' and row['customer_id'] != 'TEST':
        month = row['order_date'][:7]
        clean_revenue[month] = clean_revenue.get(month, 0) + row['amount_num']

for month in sorted(clean_revenue):
    print(f'{month}: {clean_revenue[month]:,.0f} ₸')
clean_change = (clean_revenue['2026-10'] - clean_revenue['2026-09']) / clean_revenue['2026-09'] * 100
print(f'Изменение октября к сентябрю после очистки: {clean_change:.2f}%')

customer_orders = {}
for row in unique_rows:
    customer = row['customer_id']
    customer_orders[customer] = customer_orders.get(customer, 0) + 1
most_orders = max(customer_orders, key=customer_orders.get)
print(f'Больше всего заказов у {most_orders}: {customer_orders[most_orders]}')


Полных дублей: 180
Дубли по месяцам: {'2026-09': 180}
2026-09: 35,243,410 ₸
2026-10: 34,419,520 ₸
Изменение октября к сентябрю после очистки: -2.34%
Больше всего заказов у C01818: 10
